#### **HOW TO USE THIS PROJECT TO GENERATE SYNTHETIC IMAGES ?**
1. Place this notebook in a separate folder
2. Choose the number of synthetic images you want to generate
3. Specify the prefix for your generated synthetic images (its usually the class name)
4. Specify the **"relative-path"** to the generator for that particular class
5. Run all cells
6. Your generated images should be stored at **"./generated_images/"**

In [14]:
num_images = 1991
class_name = 'AD'
path_to_generator = '../synthetic_localised/AD/best_gen/MoI_gen_422_(5.89).h30'

#### **IMPORTING NECESSARY DEPENDENCIES**

In [15]:
import os
import cv2
import shutil
import numpy as np
import seaborn as sns
from tqdm import tqdm
from numpy import cov
from PIL import Image
from numpy import trace
import tensorflow as tf
from numpy import asarray
from tensorflow import keras
from scipy.linalg import sqrtm
from numpy import iscomplexobj
import matplotlib.pyplot as plt
from numpy.random import randint
from tensorflow.keras import layers
from skimage.transform import resize
from IPython.display import FileLink
from keras.datasets.mnist import load_data
from keras.applications.inception_v3 import InceptionV3
from skimage.metrics import structural_similarity as ssim
from keras.applications.inception_v3 import preprocess_input
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

#### **DEFINING IMPORTANT VARIABLES AND PATHS**

In [24]:
BATCH_SIZE = 32
noise_dim = 256
working_directory = '../synthetic_localised/generated_images/'
os.makedirs(working_directory, exist_ok=True)
generator = tf.keras.models.load_model(path_to_generator)

#### **DEFINING IMPORTANT FUNCTIONS**

In [25]:
def clear_workspace():
    for filename in os.listdir(working_directory):
        file_path = os.path.join(working_directory, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print('Failed to delete %s. Reason: %s' % (file_path, e))

In [26]:
def download_synthetic_images(generator, number_of_images, class_name='None'):
    clear_workspace()
    
    # Define the working directory and images folder
    folder_path = os.path.join(working_directory, class_name)
    os.makedirs(folder_path, exist_ok=True)

    # Get existing images count
    existing_images = len(os.listdir(folder_path))
    
    # Initialize tqdm with existing count
    pbar = tqdm(total=number_of_images, initial=existing_images, position=0, leave=True)

    counter = existing_images + 1  # Continue numbering from the last saved image

    while counter <= number_of_images:
        remaining_images = number_of_images - existing_images
        batch_size = min(BATCH_SIZE, remaining_images)  # Avoid generating excess images

        # Generate synthetic images
        random_latent_vectors = tf.random.normal(shape=(batch_size, noise_dim))
        synthetic_images_batch = generator(random_latent_vectors).numpy()

        for image in synthetic_images_batch:
            if counter > number_of_images:  # Stop early if the required count is reached
                break  

            # Normalize and convert to uint8
            image = ((image * 127.5) + 127.5).astype(np.uint8)

            # Define image save path with .png extension
            image_path = os.path.join(folder_path, f"{class_name} ({counter}).png")

            if cv2.imwrite(image_path, image):
                pbar.update(1)  # Update progress bar only on successful save
                counter += 1
            else:
                print(f"Failed to save: {image_path}")

    pbar.close()  # Ensure tqdm closes properly
    return 'GENERATED AND SAVED ALL IMAGES'

#### **GENERATING AND SAVING SYNTHETIC IMAGES**

In [27]:
download_synthetic_images(generator, number_of_images = num_images, class_name = class_name)

100%|██████████████████████████████████████| 1991/1991 [00:06<00:00, 287.49it/s]


'GENERATED AND SAVED ALL IMAGES'